# Modèle — du socle v1 au réglage par Optuna

Premier modèle sur les **48 features** issues de la base, sans clustering et
sans lissage bayésien. Puis la même architecture, avec les hyperparamètres
choisis par Optuna : le **v2**.

Le rôle du v1 n'est pas d'être bon, **c'est de servir de point de comparaison**.
Sans lui, on ne saura jamais ce que le clustering aura apporté.

---

### Le protocole

| | |
|---|---|
| Entraînement | train 2006-2019, **échantillonné 1:10** — 368 826 lignes, 9,12 % de positifs |
| Évaluation | validation 2020-2022, **intégrale** — 38 068 464 lignes, 0,0241 % de positifs |
| Métrique | **PR-AUC** — l'accuracy est inutilisable à ce niveau de rareté |
| Modèles | RandomForest, XGBoost v1 (réglé à la main), XGBoost v2 (réglé par Optuna) |

⚠️ **Deux colonnes ont été exclues** : `nb_feux` et `surface_m2`. Vérifié,
`nb_feux > 0` égale `y` sur **100 %** des lignes — ce sont les cibles
déguisées. Les laisser aurait donné une PR-AUC proche de 1,00 et un modèle
sans aucune valeur.

⚠️ **Optuna n'a jamais vu la validation.** Il a cherché sur un découpage
*interne au train* (ajustement 2006-2017, évaluation 2018-2019). La validation
2020-2022 sert déjà à la calibration ; l'utiliser aussi pour choisir les
hyperparamètres reviendrait à la consommer deux fois.

## 1. Chargement des résultats

In [ ]:
# ── enregistrement automatique des figures ──
# chaque plt.show() écrit aussi un PNG dans figures/modele-v1/
import sys
from pathlib import Path

for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "src" / "tvfed").is_dir():
        sys.path.insert(0, str(_p / "src"))
        break

from tvfed.figures import activer

activer("modele-v1")

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

for p in (Path.cwd(), *Path.cwd().parents):
    if (p / "src" / "tvfed").is_dir():
        sys.path.insert(0, str(p / "src"))
        RACINE = p
        break

# charte identique aux notebooks d'audit
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"
BLEU, ORANGE, ROUGE, VERT, VIOLET = "#2a78d6", "#eb6834", "#e34948", "#1baf7a", "#4a3aa7"
GRIS = "#c3c2b7"
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
                     "font.size": 9, "axes.edgecolor": "#c3c2b7", "text.color": INK})

PROC = RACINE / "data" / "processed"
BASE = pd.read_csv(PROC / "baselines.csv")
MOD = pd.read_csv(PROC / "modeles_v1.csv")
IMP = pd.read_csv(PROC / "importances_v1.csv")

# les six blocs thématiques de features
BLOCS = {
    "historique commune": (["feux_commune_7j", "feux_commune_30j", "feux_commune_90j",
                            "feux_commune_365j", "jours_depuis_dernier_feu"], VIOLET),
    "météo": (["fwi", "ffmc", "dmc", "dc", "bui", "isi", "kbdi", "erc",
               "danger_effis", "fwi_j1", "ffmc_j1"], ORANGE),
    "géographie": (["lat", "lon", "distance_cote_km", "altitude_moy",
                    "amplitude_altitude", "log_superficie"], BLEU),
    "végétation": ([c for c in IMP.feature if c.startswith("part_")] + ["clc_millesime"], VERT),
    "présence humaine": (["log_population", "log_densite", "grille_densite"], ROUGE),
}


def bloc_de(f):
    for nom, (feats, _) in BLOCS.items():
        if f in feats:
            return nom
    return "calendrier"


COUL_BLOC = {n: c for n, (_, c) in BLOCS.items()} | {"calendrier": GRIS}
IMP["bloc"] = IMP.feature.map(bloc_de)

TAUX_VAL = 0.0241 / 100  # taux de positifs de la validation

print(f"baselines : {len(BASE)}   modèles : {len(MOD)}   features : {len(IMP)}")
print(f"meilleure baseline : {BASE.pr_auc.max():.4f}")
print(f"meilleur modèle    : {MOD.pr_auc.max():.4f}")

## 2. Performance

Le lift — combien de fois mieux que le hasard — est plus lisible que la PR-AUC
brute, et il se compare d'un problème à l'autre.

**Un repère indispensable pour la lecture** : la PR-AUC d'un modèle au hasard
vaut *exactement* le taux de positifs, soit 0,000241 ici. C'est la « valeur au
repos ». Tout ce qui dépasse est de l'information réelle.

In [ ]:
"""FIG 1 — Le gain, étape par étape : du hasard au modèle optimisé."""
V2 = pd.read_csv(PROC / "modeles_v2.csv")
ETAPES = pd.DataFrame({
    "nom": ["Hasard", "Danger EFFIS\n(météo seule)", "Historique commune\n(spatial seul)",
            "Historique × EFFIS\n(croisement naïf)", "RandomForest\n(48 features)",
            "XGBoost v1\n(réglé à la main)", "XGBoost v2\n(réglé par Optuna)"],
    "pr_auc": [BASE.pr_auc[0], BASE.pr_auc[2], BASE.pr_auc[1], BASE.pr_auc[3],
               MOD.set_index("modele").pr_auc["RandomForest"],
               MOD.set_index("modele").pr_auc["XGBoost"],
               V2.pr_auc[0]],
    "type": ["repos", "baseline", "baseline", "baseline", "modele", "modele", "optimise"],
})
ETAPES["lift"] = ETAPES.pr_auc / TAUX_VAL
COUL = {"repos": GRIS, "baseline": "#86b6ef", "modele": BLEU, "optimise": VIOLET}

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# (a) progression en lift
y = np.arange(len(ETAPES))
ax[0].barh(y, ETAPES.lift, color=[COUL[t] for t in ETAPES.type],
           edgecolor="#fcfcfb", linewidth=1.2, height=.72)
for i, (l, p) in enumerate(zip(ETAPES.lift, ETAPES.pr_auc)):
    ax[0].text(l + 1.5, i, f"{l:.0f}×", va="center", fontsize=10, weight="bold")
    ax[0].text(l + 9, i, f"PR-AUC {p:.4f}", va="center", fontsize=8, color=MUTED)
ax[0].set_yticks(y); ax[0].set_yticklabels(ETAPES.nom, fontsize=8.5)
ax[0].invert_yaxis()
ax[0].set_xlabel("lift — combien de fois mieux que le hasard")
ax[0].set_xlim(0, ETAPES.lift.max() * 1.42)
ax[0].set_title("Chaque source d'information ajoute son gain",
                fontsize=11.5, weight="bold", loc="left")
ax[0].axvline(1, color=INK, lw=1.2, ls="--")
ax[0].text(1.8, len(ETAPES) - .35, "valeur au repos", fontsize=8, color=MUTED)

# (b) ce que le modèle ajoute par-dessus la meilleure baseline
ref = BASE.pr_auc.max()
gain = pd.DataFrame({
    "nom": ["Meilleure\nbaseline", "RandomForest", "XGBoost v1", "XGBoost v2\n(Optuna)"],
    "v": [ref, MOD.set_index("modele").pr_auc["RandomForest"],
          MOD.set_index("modele").pr_auc["XGBoost"], V2.pr_auc[0]],
})
b = ax[1].bar(gain.nom, gain.v, color=["#86b6ef", "#5598e7", BLEU, VIOLET],
              edgecolor="#fcfcfb", linewidth=1.2, width=.62)
ax[1].axhline(ref, color=INK, ls="--", lw=1.2)
# placée dans le creux entre deux barres, SOUS la ligne : à droite elle était
# rognée, au-dessus elle heurtait l'étiquette de valeur
ax[1].text(-.55, ref * .95, "seuil à battre", va="top", ha="left",
           fontsize=8.5, color=MUTED)
for r, v in zip(b, gain.v):
    ax[1].text(r.get_x() + r.get_width() / 2, v + gain.v.max() * .025,
               f"{v:.4f}", ha="center", fontsize=10, weight="bold")
    if v > ref:
        ax[1].text(r.get_x() + r.get_width() / 2, v * .5,
                   f"×{v / ref:.2f}\nvs baseline", ha="center", fontsize=9,
                   color="#fcfcfb", weight="bold")
ax[1].set_ylabel("PR-AUC sur la validation intégrale")
ax[1].set_ylim(0, gain.v.max() * 1.18)
ax[1].set_title("Le modèle bat le croisement naïf des deux sources",
                fontsize=11.5, weight="bold", loc="left")

for a in ax:
    a.grid(axis="x" if a is ax[0] else "y", color=GRID, lw=.7)
    a.set_axisbelow(True)
    a.spines[["top", "right"]].set_visible(False)
    a.tick_params(colors=MUTED)
fig.suptitle("Performance — validation 2020-2022, 38 M lignes, 9 176 feux (0,0241 %)",
             fontsize=13, weight="bold", x=.005, ha="left", y=1.02)
plt.tight_layout(); plt.show()

print(f"Le hasard vaut exactement le taux de positifs : {TAUX_VAL:.5f}")
print(f"XGBoost = {MOD.pr_auc.max() / TAUX_VAL:.0f}× le hasard, "
      f"{MOD.pr_auc.max() / ref:.2f}× la meilleure baseline.")


## 3. D'où vient l'information ?

Les importances brutes sont trompeuses : les 11 indices météo sont corrélés
entre eux à r > 0,85, donc chacun pris isolément paraît négligeable alors que
le **bloc** pèse lourd. On les regroupe par thème.

⚠️ RandomForest et XGBoost donnent des réponses très différentes. C'est la
faiblesse connue de l'importance par impureté — instable, et biaisée vers les
variables continues à forte cardinalité. **Ces chiffres sont indicatifs ;
SHAP tranchera.**

In [ ]:
"""FIG 2 — D'où vient l'information ? Importances regroupées par bloc."""
# Les 11 indices météo sont fortement corrélés entre eux (r > 0,85) : pris
# isolément, chacun paraît négligeable alors que le BLOC pèse lourd. On somme.
g = (IMP.groupby("bloc")[["xgb", "rf"]].sum() * 100).sort_values("xgb", ascending=True)
g["n"] = IMP.groupby("bloc").size()

fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))

y = np.arange(len(g))
h = .38
ax[0].barh(y + h / 2, g.xgb, h, color=[COUL_BLOC[b] for b in g.index],
           edgecolor="#fcfcfb", linewidth=.8)
ax[0].barh(y - h / 2, g.rf, h, color=[COUL_BLOC[b] for b in g.index],
           edgecolor="#fcfcfb", linewidth=.8, alpha=.45)
for i, (x1, x2) in enumerate(zip(g.xgb, g.rf)):
    ax[0].text(x1 + .8, i + h / 2, f"{x1:.0f} %", va="center", fontsize=9, weight="bold")
    ax[0].text(x2 + .8, i - h / 2, f"{x2:.0f} %", va="center", fontsize=8, color=MUTED)
ax[0].set_yticks(y)
ax[0].set_yticklabels([f"{b}\n({int(n)} features)" for b, n in zip(g.index, g.n)],
                      fontsize=8.5)
ax[0].set_xlabel("part de l'importance totale (%)")
ax[0].set_xlim(0, max(g.xgb.max(), g.rf.max()) * 1.22)
ax[0].set_title("Importance par bloc thématique", fontsize=11.5, weight="bold", loc="left")
# la couleur code le BLOC, l'opacité code le MODÈLE : la légende doit être
# neutre, sinon elle laisse croire que le rouge désigne XGBoost
ax[0].legend([plt.Rectangle((0, 0), 1, 1, color=MUTED),
              plt.Rectangle((0, 0), 1, 1, color=MUTED, alpha=.45)],
             ["XGBoost", "RandomForest"], frameon=False, fontsize=9,
             loc="lower right")

# les 15 features les plus importantes, colorées par bloc
top = IMP.nlargest(15, "xgb").sort_values("xgb")
ax[1].barh(top.feature, top.xgb * 100, color=[COUL_BLOC[b] for b in top.bloc],
           edgecolor="#fcfcfb", linewidth=.8)
ax[1].set_xlabel("importance XGBoost (%)")
ax[1].set_title("Les 15 features les plus utilisées", fontsize=11.5, weight="bold", loc="left")
ax[1].tick_params(axis="y", labelsize=8)
poignees = [plt.Rectangle((0, 0), 1, 1, color=COUL_BLOC[b]) for b in g.index[::-1]]
ax[1].legend(poignees, list(g.index[::-1]), frameon=False, fontsize=7.5,
             loc="lower right", title="bloc", title_fontsize=8)

for a in ax:
    a.grid(axis="x", color=GRID, lw=.7); a.set_axisbelow(True)
    a.spines[["top", "right"]].set_visible(False); a.tick_params(colors=MUTED)
fig.suptitle("Le modèle est à moitié un modèle de persistance",
             fontsize=13, weight="bold", x=.005, ha="left", y=1.03)
plt.tight_layout(); plt.show()

hist = g.loc["historique commune"]
print(f"Le bloc « historique commune » pèse {hist.xgb:.0f} % pour XGBoost "
      f"et {hist.rf:.0f} % pour RandomForest.")
print()
print("⚠️  Les deux modèles sont en net DÉSACCORD. C'est la faiblesse connue de")
print("    l'importance par impureté : instable, et biaisée vers les variables")
print("    continues à forte cardinalité. Ces chiffres sont indicatifs —")
print("    SHAP tranchera, et c'est lui qui ira dans le rapport.")
print()
print("→ Conséquence de fond : le modèle prédit surtout que « les communes qui")
print("  ont brûlé rebrûleront ». Une commune sans historique reste condamnée")
print("  à un score bas — c'est le problème de small area estimation, et c'est")
print("  précisément ce que le clustering spatial doit corriger.")

## 4. Précision, rappel, et ce que ça donne concrètement

Le modèle donne une **note de risque** à chacun des 38 millions de couples
commune × jour. On les trie du plus risqué au moins risqué, puis on décide
d'en surveiller une certaine part — parce qu'aucun service n'a les moyens de
surveiller tout le territoire tous les jours.

Deux mesures répondent à deux questions différentes :

| | Question | |
|---|---|---|
| **Rappel** | sur 100 feux réels, combien sont dans la zone surveillée ? | on veut qu'il monte |
| **Précision** | sur 100 alertes émises, combien étaient un vrai feu ? | on veut qu'elle monte aussi |

Elles s'opposent : surveiller large attrape plus de feux mais déclenche
beaucoup de fausses alertes.

⚠️ **La précision paraîtra catastrophique — et ce n'est pas la faute du
modèle.** Sur 38 millions de couples il n'y a que 9 176 feux. En surveillant
1 % du territoire-jour, soit 380 000 couples, même un modèle **parfait** ne
pourrait viser juste que 9 176 fois : 2,4 % de précision au maximum. C'est de
l'arithmétique, pas de la performance. Le second graphique montre ce plafond.

In [ ]:
"""FIG 3 — Précision et rappel, en clair.

Le modèle donne une note de risque à chacun des 38 millions de couples
commune × jour de la validation. On les trie de la plus risquée à la moins
risquée, puis on décide d'en surveiller une certaine part.

Deux questions, deux mesures :

  RAPPEL     sur 100 feux réels, combien sont dans la zone surveillée ?
  PRÉCISION  sur 100 alertes émises, combien correspondent à un vrai feu ?

Elles s'opposent : plus on surveille large, plus on attrape de feux (rappel ↑)
mais plus on se trompe souvent (précision ↓).
"""
PRED = pd.read_parquet(PROC / "predictions_val.parquet")
p_xgb, y = PRED.p_xgb.to_numpy(), PRED.y.to_numpy()
taux = y.mean()

o = np.argsort(-p_xgb)
tp = np.cumsum(y[o])
k = np.arange(1, len(y) + 1)
precision, rappel, part = 100 * tp / k, 100 * tp / tp[-1], 100 * k / len(y)

# on ne trace que jusqu'à 20 % : au-delà, surveiller autant n'a aucun sens
# opérationnel, et l'écraser sur l'axe rendrait la zone utile illisible
z = part <= 20
pas = max(1, z.sum() // 3000)          # allège le tracé sans changer la courbe
pr, ra, pa = precision[z][::pas], rappel[z][::pas], part[z][::pas]

fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))

# ── (a) les deux courbes, chacune sur SON échelle ──
# Deux échelles rendent les deux formes lisibles d'un coup d'œil, mais
# ⚠️ le POINT DE CROISEMENT n'a aucun sens : il ne dépend que des bornes
# choisies. Les axes sont colorés pour qu'on sache d'emblée qui lit quoi.
ax0b = ax[0].twinx()
l1, = ax[0].plot(pa, ra, lw=2.6, color=BLEU, label="Rappel — feux attrapés")
l2, = ax0b.plot(pa, pr, lw=2.6, color=ROUGE, label="Précision — alertes justes")

ax[0].set_ylim(0, 100); ax[0].set_xlim(0, 20)
ax[0].set_ylabel("RAPPEL — % des feux attrapés", color=BLEU, weight="bold")
ax[0].tick_params(axis="y", colors=BLEU)
ax[0].spines["left"].set_color(BLEU); ax[0].spines["left"].set_linewidth(2)

ax0b.set_ylim(0, 10)
ax0b.set_ylabel("PRÉCISION — % d'alertes justes", color=ROUGE, weight="bold")
ax0b.tick_params(axis="y", colors=ROUGE)
ax0b.spines["right"].set_color(ROUGE); ax0b.spines["right"].set_linewidth(2)
ax0b.spines["top"].set_visible(False)

ax[0].set_xlabel("part des communes-jours surveillés (%)")
ax[0].set_title("Rappel et précision — deux échelles",
                fontsize=11.5, weight="bold", loc="left")
ax[0].legend(handles=[l1, l2], frameon=False, fontsize=9.5, loc="upper center")
# dans le creux entre les deux courbes : en bas à gauche l'avertissement
# passait sous la courbe rouge. Emoji retiré, matplotlib le rend mal.
ax[0].text(.50, .22, "les deux courbes n'ont pas la même échelle —\n"
                     "leur point de croisement ne signifie rien",
           transform=ax[0].transAxes, fontsize=8, color=MUTED, ha="center",
           bbox=dict(boxstyle="round,pad=0.4", facecolor="#fcfcfb",
                     edgecolor=GRID, linewidth=.8))

# ── (b) zoom sur la précision, avec son plafond mathématique ──
# Sur 38 M couples il n'y a que 9 176 feux. En surveillant 1 % (380 000
# couples), même un modèle PARFAIT ne pourrait être juste que 9 176 fois :
# soit 2,4 % de précision au maximum. Ce plafond n'est pas une limite du
# modèle, c'est l'arithmétique de la rareté.
plafond = np.minimum(100, 100 * taux / (pa / 100))
ax[1].fill_between(pa, pr, plafond, color=ROUGE, alpha=.10)
ax[1].plot(pa, plafond, lw=2, color=INK, ls="--",
           label="plafond : mieux est impossible")
ax[1].plot(pa, pr, lw=2.6, color=ROUGE, label="précision atteinte")
ax[1].set_xlabel("part des communes-jours surveillés (%)")
ax[1].set_ylabel("précision (%)")
ax[1].set_ylim(0, 10); ax[1].set_xlim(0, 5)   # l'écart au plafond se joue avant 5 %
ax[1].set_title("Zoom sur la précision", fontsize=11.5, weight="bold", loc="left")
ax[1].legend(frameon=False, fontsize=9)

# ── (c) la lecture opérationnelle, en barres ──
# « Si le service dispose de moyens pour surveiller X % du territoire-jour,
#   quelle part des feux se produit dans la zone qu'il a choisie ? »
niveaux = [0.1, 0.5, 1, 2, 5, 10]
vals = [rappel[np.argmin(np.abs(part - f))] for f in niveaux]
b = ax[2].bar(range(len(niveaux)), vals, color=BLEU, edgecolor="#fcfcfb",
              linewidth=1.2, width=.68)
for rect, v, f in zip(b, vals, niveaux):
    ax[2].text(rect.get_x() + rect.get_width() / 2, v + 2, f"{v:.0f} %",
               ha="center", fontsize=10.5, weight="bold")
    ax[2].text(rect.get_x() + rect.get_width() / 2, 3, f"×{v / f:.0f}",
               ha="center", fontsize=8.5, color="#fcfcfb", weight="bold")
ax[2].set_xticks(range(len(niveaux)))
ax[2].set_xticklabels([f"{f:g} %" for f in niveaux])
ax[2].set_xlabel("part des communes-jours surveillés")
ax[2].set_ylabel("part des feux attrapés (%)")
ax[2].set_ylim(0, 100)
ax[2].set_title("Combien de feux pour combien d'effort ?",
                fontsize=11.5, weight="bold", loc="left")
ax[2].text(.02, .97, "en blanc : le gain par rapport\nà une surveillance au hasard",
           transform=ax[2].transAxes, va="top", fontsize=8, color=MUTED)

for a in ax:
    a.grid(color=GRID, lw=.7); a.set_axisbelow(True)
    a.spines[["top", "right"]].set_visible(False); a.tick_params(colors=MUTED)
fig.suptitle("Ce que le modèle permet — validation 2020-2022, 38 M communes-jours, 9 176 feux",
             fontsize=13, weight="bold", x=.005, ha="left", y=1.03)
plt.tight_layout(); plt.show()

print("Sur les 38 068 464 couples commune × jour de la validation, 9 176 ont brûlé.")
print("Le modèle les note tous, on les trie, et on surveille les mieux notés :\n")
print(f"{'surveillé':>10s} {'= combien':>12s} {'feux attrapés':>15s} {'précision':>11s} {'vs hasard':>10s}")
for f in (0.1, 0.5, 1, 2, 5, 10):
    i = np.argmin(np.abs(part - f))
    print(f"{f:9.1f}% {int(len(y) * f / 100):12,} {rappel[i]:14.1f}% "
          f"{precision[i]:10.2f}% {rappel[i] / f:9.0f}×")
print()
print("→ Lire la 3e ligne : en surveillant 1 % du territoire-jour — soit")
print("  380 685 couples commune × jour sur 38 millions — on couvre 39 % des")
print("  feux. C'est 39 fois mieux qu'une surveillance tirée au hasard.")
print()
print("→ La précision paraît faible (0,93 %) mais elle bute sur un PLAFOND :")
print("  avec 9 176 feux seulement, un modèle PARFAIT surveillant 1 % ne")
print("  pourrait pas dépasser 2,4 % de précision. On en atteint 39 %.")

## 5. Régler les probabilités

### Le problème

Pour entraîner le modèle, on a jeté 99,8 % des jours sans feu — sinon il y en
aurait eu 400 fois trop et l'apprentissage aurait été impraticable. Le modèle
a donc appris dans un monde où les feux sont **400 fois plus fréquents** qu'en
vrai, et il annonce ses probabilités à cette échelle-là.

Résultat : il annonce **4,4 % en moyenne** alors que la réalité est
**0,024 %**. Il se trompe d'un facteur 180.

C'est un thermomètre qui affiche toujours 5 °C de trop. Il classe correctement
les jours du plus chaud au plus froid — mais ses chiffres sont inutilisables
tels quels. Il faut le régler.

### Les trois versions comparées

Ce sont **les mêmes prédictions**, à trois échelles différentes :

| | |
|---|---|
| **brut** | les probabilités telles que le modèle les sort |
| **Platt** | recalées par une courbe en S lisse, à deux paramètres |
| **isotonique** | recalées par une courbe en escalier, plus souple |

Platt et isotonique ne changent rien au modèle : ce sont deux **recettes de
réglage** appliquées à sa sortie.

⚠️ Le réglage est appris sur une moitié de la validation et vérifié sur
l'autre. L'apprendre et le montrer sur les mêmes lignes donnerait un résultat
parfait par construction, donc sans valeur.

In [ ]:
"""FIG 4 — Calibration : le modèle annonce des probabilités fausses.

LE PROBLÈME
    Pour entraîner le modèle, on a jeté 99,8 % des jours sans feu — sinon il
    y en aurait eu 400 fois trop et l'apprentissage aurait été impraticable.
    Le modèle a donc vu un monde où les feux sont 400 fois plus fréquents
    qu'en vrai, et il a appris à annoncer des probabilités à cette échelle-là.

    Résultat : il annonce 4,4 % en moyenne, alors que la réalité est 0,024 %.
    Il se trompe d'un facteur 180.

    C'est un thermomètre qui affiche toujours 5 °C de trop : il classe
    correctement les jours du plus chaud au plus froid, mais ses chiffres
    sont inutilisables tels quels. Il faut le régler.

LES TROIS COURBES
    brut        les probabilités telles que le modèle les sort
    Platt       recalées par une courbe en S à deux paramètres
    isotonique  recalées par une courbe en escalier, plus souple

    Platt et isotonique sont deux RECETTES DE RÉGLAGE différentes appliquées
    aux mêmes prédictions. Elles ne changent pas le modèle, juste l'échelle
    de ses chiffres.

⚠️ Le réglage est appris sur une moitié de la validation et vérifié sur
l'autre. L'apprendre et le montrer sur les mêmes lignes donnerait un résultat
parfait par construction, donc sans valeur.
"""
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss

rng = np.random.default_rng(42)
moitie = rng.random(len(PRED)) < .5
p_fit, y_fit = p_xgb[moitie], y[moitie]
p_ev, y_ev = p_xgb[~moitie], y[~moitie]

iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p_fit, y_fit)
p_iso = iso.predict(p_ev)

_lg = lambda v: np.log(np.clip(v, 1e-9, 1 - 1e-9) / (1 - np.clip(v, 1e-9, 1 - 1e-9))).reshape(-1, 1)
platt = LogisticRegression(C=1e10).fit(_lg(p_fit), y_fit)
p_platt = platt.predict_proba(_lg(p_ev))[:, 1]

# On découpe les communes-jours en 5 GROUPES DE RISQUE, du plus au moins
# risqué selon le modèle. Pour chacun : ce qu'on annonce vs ce qui arrive.
ordre = np.argsort(-p_ev)
n = len(p_ev)
BORNES = [0, .001, .01, .05, .20, 1.0]
NOMS_G = ["top 0,1 %", "0,1 → 1 %", "1 → 5 %", "5 → 20 %", "les 80 % restants"]
groupes = []
for i in range(5):
    sel = ordre[int(BORNES[i] * n):int(BORNES[i + 1] * n)]
    groupes.append({
        "groupe": NOMS_G[i],
        "observe": 100 * y_ev[sel].mean(),
        "brut": 100 * p_ev[sel].mean(),
        "platt": 100 * p_platt[sel].mean(),
        "iso": 100 * p_iso[sel].mean(),
    })
G = pd.DataFrame(groupes)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
x = np.arange(len(G))
w = .38

# ── (a) AVANT : ce que le modèle annonce vs ce qui arrive vraiment ──
ax[0].bar(x - w / 2, G.brut, w, color="#86b6ef", edgecolor="#fcfcfb",
          linewidth=1, label="annoncé par le modèle")
ax[0].bar(x + w / 2, G.observe, w, color=INK, edgecolor="#fcfcfb",
          linewidth=1, label="observé en réalité")
for i, (a_, o_) in enumerate(zip(G.brut, G.observe)):
    ax[0].text(i - w / 2, a_ + .4, f"{a_:.1f}", ha="center", fontsize=8.5, weight="bold")
    ax[0].text(i + w / 2, o_ + .4, f"{o_:.2f}", ha="center", fontsize=8.5)
ax[0].set_title("AVANT — le modèle annonce beaucoup trop",
                fontsize=11.5, weight="bold", loc="left")
ax[0].set_ylabel("probabilité de feu (%)")
ax[0].legend(frameon=False, fontsize=9)

# ── (b) APRÈS Platt : les deux barres doivent coïncider ──
ax[1].bar(x - w / 2, G.platt, w, color=ORANGE, edgecolor="#fcfcfb",
          linewidth=1, label="annoncé après réglage")
ax[1].bar(x + w / 2, G.observe, w, color=INK, edgecolor="#fcfcfb",
          linewidth=1, label="observé en réalité")
for i, (a_, o_) in enumerate(zip(G.platt, G.observe)):
    ax[1].text(i - w / 2, a_ + .04, f"{a_:.2f}", ha="center", fontsize=8.5, weight="bold")
    ax[1].text(i + w / 2, o_ + .04, f"{o_:.2f}", ha="center", fontsize=8.5)
ax[1].set_title("APRÈS réglage (Platt) — les barres se rejoignent",
                fontsize=11.5, weight="bold", loc="left")
ax[1].set_ylabel("probabilité de feu (%)")
ax[1].legend(frameon=False, fontsize=9)

for a in (ax[0], ax[1]):
    a.set_xticks(x)
    a.set_xticklabels(G.groupe, fontsize=8, rotation=18, ha="right")
    a.set_xlabel("groupe de risque selon le modèle")

# ── (c) le facteur d'erreur moyen, avant et après ──
biais = [p_ev.mean() / y_ev.mean(), p_platt.mean() / y_ev.mean(),
         p_iso.mean() / y_ev.mean()]
b = ax[2].bar(["brut", "Platt", "isotonique"], biais,
              color=["#86b6ef", ORANGE, VERT], edgecolor="#fcfcfb",
              linewidth=1.2, width=.55)
ax[2].axhline(1, color=INK, ls="--", lw=1.4)
for rect, v in zip(b, biais):
    ax[2].text(rect.get_x() + rect.get_width() / 2, v + 5,
               f"×{v:.0f}" if v > 10 else f"×{v:.1f}",
               ha="center", fontsize=12, weight="bold")
# placée haut : à hauteur de la ligne, elle heurtait les étiquettes ×1,4 et ×1,0
ax[2].text(1.5, max(biais) * .75, "objectif : ×1\nannoncé = observé", ha="center",
           fontsize=9.5, color=MUTED)
ax[2].set_ylabel("combien de fois le modèle sur-estime")
ax[2].set_ylim(0, max(biais) * 1.2)
ax[2].set_title("Le facteur d'erreur", fontsize=11.5, weight="bold", loc="left")

for a in ax:
    a.grid(axis="y", color=GRID, lw=.7); a.set_axisbelow(True)
    a.spines[["top", "right"]].set_visible(False); a.tick_params(colors=MUTED)
fig.suptitle("Régler les probabilités — sans quoi le score affiché serait faux d'un facteur 180",
             fontsize=13, weight="bold", x=.005, ha="left", y=1.03)
plt.tight_layout(); plt.show()

print("Ce que le modèle annonce, par groupe de risque (%) :\n")
print(G.round(3).to_string(index=False))
print()
print(f"{'':12s} {'PR-AUC':>9s} {'facteur d erreur':>18s}")
for nom, pp in [("brut", p_ev), ("Platt", p_platt), ("isotonique", p_iso)]:
    print(f"{nom:12s} {average_precision_score(y_ev, pp):9.4f} "
          f"{pp.mean() / y_ev.mean():17.1f}×")
print()
print("→ Les deux réglages ramènent le facteur d'erreur de 180 à ~1.")
print()
print("⚠️  MAIS ils ne se valent pas sur le CLASSEMENT :")
print(f"   Platt garde la PR-AUC intacte ({average_precision_score(y_ev, p_platt):.4f}),")
print(f"   l'isotonique la fait tomber à {average_precision_score(y_ev, p_iso):.4f}, soit −10 %.")
print()
print("   Pourquoi : l'isotonique est une courbe EN ESCALIER. Elle regroupe")
print(f"   des millions de scores différents en {len(np.unique(p_iso)):,} paliers seulement,")
print("   et à l'intérieur d'un palier tout le monde se retrouve à égalité —")
print("   l'ordre y est perdu. Platt est une courbe lisse : elle ne crée")
print("   aucune égalité, donc elle conserve le classement exactement.")

## 6. Combien d'arbres faut-il vraiment ?

Avant de lancer une recherche d'hyperparamètres, une question plus simple :
le v1 en construisait **400**. Est-ce trop, pas assez, ou au bon endroit ?

XGBoost construit ses arbres **en séquence**, chacun corrigeant les erreurs
des précédents. On peut donc mesurer la performance après 1 arbre, après 2,
… après 600, et voir exactement quand ça cesse de progresser.

La mesure se fait sur le **découpage interne au train** (ajustement 2006-2017,
évaluation 2018-2019) — la validation reste intacte.

In [ ]:
"""FIG 5 — La courbe d'apprentissage : combien d'arbres faut-il ?

XGBoost construit ses arbres EN SÉQUENCE, chacun corrigeant les erreurs des
précédents. On peut donc mesurer la performance après 1 arbre, après 2, …
après 600, et voir quand ça cesse de progresser.

Deux courbes, deux jeux de données :
  AJUSTEMENT  les lignes sur lesquelles le modèle apprend   (2006-2017)
  ÉVALUATION  des lignes qu'il n'a jamais vues              (2018-2019)

Le modèle progresse forcément sur l'ajustement — c'est là qu'il apprend.
La seule courbe qui compte est celle de l'évaluation.
"""
C = pd.read_csv(PROC / "courbe_apprentissage.csv")
best = int(C.evaluation.idxmax()) + 1
plateau = int(C.index[C.evaluation >= .99 * C.evaluation.max()][0]) + 1

fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))

# ── (a) les deux courbes ──
ax[0].plot(C.iteration, C.ajustement, lw=2.4, color="#86b6ef",
           label="Ajustement (2006-2017) — ce qu'il a appris")
ax[0].plot(C.iteration, C.evaluation, lw=2.6, color=BLEU,
           label="Évaluation (2018-2019) — ce qui compte")
ax[0].axvline(plateau, color=VERT, ls="--", lw=1.6)
ax[0].axvline(400, color=MUTED, ls=":", lw=1.6)
ax[0].text(plateau + 12, .60, f"plateau atteint\ndès {plateau} arbres",
           fontsize=8.5, color=VERT, weight="bold")
ax[0].text(410, .93, "400 : le réglage\nutilisé pour le v1", fontsize=8.5, color=MUTED)
ax[0].set_xlabel("nombre d'arbres construits")
ax[0].set_ylabel("PR-AUC")
ax[0].set_ylim(.55, 1.0)
ax[0].set_title("Le modèle progresse-t-il encore ?", fontsize=11.5, weight="bold", loc="left")
ax[0].legend(frameon=False, fontsize=9, loc="lower right")

# ── (b) l'écart entre les deux : la mesure du sur-apprentissage ──
ecart = C.ajustement - C.evaluation
ax[1].fill_between(C.iteration, 0, ecart, color=ROUGE, alpha=.18)
ax[1].plot(C.iteration, ecart, lw=2.4, color=ROUGE)
ax[1].axvline(plateau, color=VERT, ls="--", lw=1.6)
ax[1].set_xlabel("nombre d'arbres construits")
ax[1].set_ylabel("écart ajustement − évaluation")
ax[1].set_title("Le sur-apprentissage, mesuré", fontsize=11.5, weight="bold", loc="left")
ax[1].annotate(f"au-delà de {plateau} arbres, le modèle\n"
               "apprend par cœur sans mieux généraliser",
               xy=(500, ecart.iloc[499]), xytext=(200, ecart.max() * .55),
               fontsize=8.5, color=ROUGE,
               arrowprops=dict(arrowstyle="->", color=ROUGE, lw=1.2))

for a in ax:
    a.grid(color=GRID, lw=.7); a.set_axisbelow(True)
    a.spines[["top", "right"]].set_visible(False); a.tick_params(colors=MUTED)
fig.suptitle("Courbe d'apprentissage — mesurée sur un découpage interne au train, "
             "la validation reste intacte",
             fontsize=12.5, weight="bold", x=.005, ha="left", y=1.03)
plt.tight_layout(); plt.show()

print(f"Meilleure évaluation : {C.evaluation.max():.4f} à l'itération {best}")
print(f"99 % de ce maximum atteint dès l'itération {plateau}")
print(f"Écart final (600 arbres) : {ecart.iloc[-1]:.4f}\n")
print("→ Les 200 derniers arbres n'apportent RIEN en généralisation.")
print("  C'est exactement le paramètre qu'Optuna doit arbitrer, en le")
print("  combinant au learning_rate et à la profondeur.")
print()
print("⚠️  Ces PR-AUC (~0,73) ne se comparent PAS aux 0,0166 de la validation :")
print("    on est ici sur le train échantillonné, où les positifs sont 350 fois")
print("    plus fréquents qu'en réalité. Ce sont des valeurs COMPARATIVES entre")
print("    configurations, jamais des valeurs absolues.")


## 7. L'optimisation des hyperparamètres

Un hyperparamètre est un réglage qu'on **choisit avant** l'entraînement :
la profondeur des arbres, la vitesse d'apprentissage, la force de la
régularisation. Le modèle ne les apprend pas, il les subit.

Le v1 les avait fixés à la main, sur des valeurs raisonnables. **Optuna**
teste des combinaisons et apprend au fil des essais où chercher — c'est
l'échantillonnage *TPE*, qui resserre la recherche autour des zones
prometteuses au lieu de balayer une grille aveugle.

**60 essais**, 9 hyperparamètres, sur le découpage interne au train.

In [ ]:
"""FIG 6 — Ce qu'Optuna a exploré, et ce qu'il en ressort.

Optuna teste des combinaisons d'hyperparamètres et apprend au fil des essais
où chercher (échantillonnage TPE, pas une grille aveugle). Trois questions :

  1. la recherche a-t-elle convergé, ou faudrait-il plus d'essais ?
  2. quel hyperparamètre compte réellement ?
  3. le gain vaut-il le temps passé ?
"""
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
etude = optuna.load_study(study_name="tvfed_xgb",
                          storage=f"sqlite:///{(PROC / 'optuna.db').as_posix()}")
E = etude.trials_dataframe()
E = E[E.state == "COMPLETE"].reset_index(drop=True)
E["meilleur"] = E.value.cummax()
REF_MANUEL = 0.7340          # le réglage à la main, mesuré au même endroit

fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))

# ── (a) l'historique : chaque point un essai, la ligne le meilleur atteint ──
ax[0].scatter(E.number + 1, E.value, s=26, color="#86b6ef",
              edgecolors="none", label="essais")
ax[0].plot(E.number + 1, E.meilleur, lw=2.6, color=BLEU, label="meilleur atteint")
ax[0].axhline(REF_MANUEL, color=ORANGE, ls="--", lw=1.8)
ax[0].text(len(E) * .45, REF_MANUEL - .006, "réglage manuel du v1",
           fontsize=8.5, color=ORANGE, weight="bold")
ax[0].set_xlabel("numéro d'essai")
ax[0].set_ylabel("PR-AUC (découpage interne au train)")
ax[0].set_ylim(max(0, E.value.quantile(.05) - .01), E.value.max() + .008)
ax[0].set_title("La recherche converge-t-elle ?", fontsize=11.5, weight="bold", loc="left")
ax[0].legend(frameon=False, fontsize=9, loc="lower right")

# ── (b) quel hyperparamètre compte ? ──
try:
    imp = optuna.importance.get_param_importances(etude)
    s = pd.Series(imp).sort_values()
    ax[1].barh(s.index, 100 * s.values, color=VIOLET,
               edgecolor="#fcfcfb", linewidth=.8)
    for i, v in enumerate(100 * s.values):
        ax[1].text(v + .8, i, f"{v:.0f} %", va="center", fontsize=8.5, weight="bold")
    ax[1].set_xlim(0, 100 * s.max() * 1.2)
except Exception as e:                     # étude trop courte pour l'analyse
    ax[1].text(.5, .5, f"importances indisponibles\n({type(e).__name__})",
               ha="center", transform=ax[1].transAxes, color=MUTED)
ax[1].set_xlabel("part de la variation expliquée (%)")
ax[1].set_title("Quel réglage compte vraiment ?", fontsize=11.5, weight="bold", loc="left")
ax[1].tick_params(axis="y", labelsize=8)

# ── (c) le gain d'Optuna : mesuré ici, puis mesuré pour de vrai ──
# ⚠️ Le point le plus instructif de la figure. Le découpage interne annonçait
# +0,7 % ; sur la validation réelle le même modèle gagne +5,2 %. Le proxy a
# sous-estimé d'un facteur ~8 — parce qu'à 8 % de positifs (train échantillonné)
# et à 0,024 % (réalité), ce n'est pas la même partie du classement qui compte.
gain_interne = 100 * (E.value.max() / REF_MANUEL - 1)
V2 = pd.read_csv(PROC / "modeles_v2.csv")
V1 = pd.read_csv(PROC / "modeles_v1.csv").set_index("modele")
gain_reel = 100 * (V2.pr_auc[0] / V1.pr_auc["XGBoost"] - 1)

mes = pd.DataFrame({
    "ou": ["Mesuré pendant\nla recherche\n(découpage interne)",
           "Mesuré après coup\nsur la validation\n(38 M lignes réelles)"],
    "gain": [gain_interne, gain_reel],
})
b = ax[2].bar(mes.ou, mes.gain, color=["#c9c4de", VIOLET],
              edgecolor="#fcfcfb", linewidth=1.2, width=.55)
for rect, v in zip(b, mes.gain):
    ax[2].text(rect.get_x() + rect.get_width() / 2, v + .12, f"+{v:.1f} %",
               ha="center", fontsize=13, weight="bold")
ax[2].set_ylabel("gain d'Optuna sur le v1 (%)")
ax[2].set_ylim(0, gain_reel * 1.42)
ax[2].set_title("Le gain, avant et après vérification",
                fontsize=11.5, weight="bold", loc="left")
ax[2].text(.5, .985, f"le découpage interne a SOUS-ESTIMÉ le gain\n"
                     f"d'un facteur {gain_reel / gain_interne:.0f} — voir la lecture ci-dessous",
           transform=ax[2].transAxes, ha="center", va="top",
           fontsize=8.5, color=MUTED)

for a in ax:
    a.grid(color=GRID, lw=.7); a.set_axisbelow(True)
    a.spines[["top", "right"]].set_visible(False); a.tick_params(colors=MUTED)
ax[1].grid(axis="y", visible=False)
fig.suptitle(f"Optimisation Optuna — {len(E)} essais, échantillonnage TPE",
             fontsize=13, weight="bold", x=.005, ha="left", y=1.03)
plt.tight_layout(); plt.show()

print(f"{len(E)} essais menés à terme")
print(f"meilleur   : {E.value.max():.4f}  (essai {int(E.value.idxmax()) + 1})")
print(f"réglage manuel du v1 : {REF_MANUEL:.4f}")
print(f"gain mesuré PENDANT la recherche : {gain_interne:+.2f} %\n")
print("Hyperparamètres retenus :")
for k, v in etude.best_params.items():
    print(f"   {k:20s} {v}")
print()
print("─" * 66)
print("CE QU'IL FAUT RETENIR")
print("─" * 66)
print(f"Pendant la recherche, Optuna annonçait {gain_interne:+.2f} % — négligeable.")
print(f"Réentraîné et mesuré sur la validation réelle : {gain_reel:+.1f} %.")
print(f"Le découpage interne a sous-estimé le vrai gain d'un facteur "
      f"{gain_reel / gain_interne:.0f}.")
print()
print("Pourquoi ? Les deux mesures ne portent pas sur la même population.")
print("  · découpage interne : train échantillonné, ~8 % de positifs")
print("  · validation réelle : 38 M lignes, 0,024 % de positifs")
print("À 8 % de positifs, la PR-AUC dépend de tout le classement. À 0,024 %,")
print("elle ne dépend pratiquement que de l'extrême haut du classement — les")
print("quelques milliers de lignes les mieux notées. Optuna a choisi un modèle")
print("plus lent et plus régularisé (900 arbres, lr 0,012, min_child_weight 25)")
print("qui trie mieux CE haut de classement, ce que le proxy voyait à peine.")
print()
print("→ Leçon de méthode : un score de recherche est un OUTIL DE COMPARAISON")
print("  entre configurations, pas une prédiction du gain final. Seul le")
print("  réentraînement complet suivi de la mesure sur validation fait foi.")


## 8. Synthèse

### Les scores

| Prédicteur | PR-AUC | lift | connaît |
|---|---|---|---|
| Hasard | 0,0002 | 1,0× | rien |
| Danger EFFIS | 0,0012 | 5,1× | le *quand* |
| Historique commune | 0,0047 | 19,4× | le *où* |
| Historique × EFFIS | 0,0101 | 42,1× | les deux, croisés naïvement |
| RandomForest | 0,0150 | 62,0× | 48 features |
| XGBoost v1 | 0,0166 | 68,8× | 48 features, réglé à la main |
| **XGBoost v2** | **0,0175** | **72,4×** | 48 features, réglé par Optuna |

**Le modèle bat la meilleure baseline de ×1,72.** Aucun signal de fuite : on
est très loin du seuil d'alerte de 0,80, et aucune feature seule ne dépasse
0,49 sur le train.

⚠️ **Piège de lecture** : le cadrage annonçait « un bon modèle fait 0,20-0,35 ».
Ce chiffre valait pour un événement à 0,2 %. Le nôtre est à **0,024 %**, dix
fois plus rare, et la PR-AUC décroît mécaniquement avec la rareté. La grandeur
comparable est le **lift**.

### Le résultat le plus instructif de la section 7

| Où le gain d'Optuna a été mesuré | Gain annoncé |
|---|---|
| pendant la recherche — découpage interne, train échantillonné, ~8 % de positifs | **+0,7 %** |
| après coup — validation intégrale, 38 M lignes, 0,024 % de positifs | **+5,2 %** |

Le proxy de recherche a **sous-estimé le gain réel d'un facteur 8**, et il
faut comprendre pourquoi : les deux mesures ne portent pas sur la même
population. À 8 % de positifs, la PR-AUC dépend de tout le classement ; à
0,024 %, elle ne dépend pratiquement que de son **extrême sommet**. Optuna a
retenu un modèle plus lent et plus régularisé (900 arbres, `learning_rate`
0,012, `min_child_weight` 25) qui trie mieux ce sommet — ce que le proxy ne
voyait presque pas.

→ **Un score de recherche est un outil de comparaison entre configurations,
pas une prédiction du gain final.** Seul le réentraînement complet suivi de la
mesure sur validation fait foi. C'est vrai dans les deux sens : ici le gain
était meilleur qu'annoncé, il aurait tout aussi bien pu être pire.

Second enseignement : **`learning_rate` explique 91 % de la variation** entre
essais. Les 8 autres hyperparamètres réunis pèsent moins de 10 %. Une
recherche sur ce seul paramètre aurait capté l'essentiel du gain.

### Ce que le modèle a appris — et sa limite

| Bloc | XGBoost | RandomForest |
|---|---|---|
| **historique de la commune** | **54,6 %** | 26,9 % |
| météo | 16,2 % | 27,8 % |
| géographie | 13,1 % | 19,1 % |
| végétation | 8,5 % | 17,7 % |
| calendrier | 6,3 % | 5,5 % |
| présence humaine | 1,3 % | 3,0 % |

**Le modèle est à moitié un modèle de persistance** : il dit surtout que « les
communes qui ont brûlé rebrûleront ». Ce n'est pas une fuite — le jour J on
connaît réellement l'historique des 365 jours précédents — mais ça a deux
conséquences :

1. **Ça explique le gain modeste.** La baseline 1 faisait déjà exactement ça.
2. **Une commune sans historique reste condamnée à un score bas.** C'est le
   problème de *small area estimation* : une commune qui n'a jamais brûlé mais
   qui est entourée de communes qui brûlent devrait être signalée, et elle ne
   l'est pas.

→ **C'est l'hypothèse à tester avec le clustering spatial** : s'il apporte
quelque chose, ce sera d'abord sur ces communes-là.

### La calibration : un arbitrage réel

| | PR-AUC | Brier | biais de niveau | valeurs distinctes |
|---|---|---|---|---|
| brut | 0,0162 | 1,70·10⁻² | ×181 | 12 306 930 |
| **Platt** | **0,0162** | 2,394·10⁻⁴ | ×1,4 | 7 848 276 |
| **isotonique** | 0,0146 | 2,393·10⁻⁴ | **×1,0** | **52 147** |

Les deux corrigent le biais et divisent le Brier par ~70. Mais **l'isotonique
fait chuter la PR-AUC de 10 %** : c'est une fonction *en escalier*, elle écrase
des millions de scores distincts en 52 000 paliers, et l'ordre à l'intérieur
d'un palier est définitivement perdu. Platt est une sigmoïde strictement
croissante : elle préserve l'ordre exactement.

Le choix dépend de l'usage — l'application fait les deux (carte classée **et**
score affiché). À trancher après SHAP.

### Où en est le budget de gain

| Levier | État | Gain constaté |
|---|---|---|
| Features de la base | fait | ×1,63 sur la meilleure baseline |
| Hyperparamètres (Optuna) | fait | +5,2 % de plus |
| **Clustering spatial** | **à faire** | référence à battre : **0,0175** |
| MLP | à faire | troisième famille exigée par l'énoncé |
| SHAP | à faire | interprétation, pas performance |

### Prochaines étapes

1. **Clustering spatial**, fitté sur le train seul → mesurer l'écart à **0,0175**
   (et non plus 0,0166 : le v2 est la nouvelle référence)
2. **MLP**, la troisième famille exigée par l'énoncé
3. **SHAP** — avec la matrice de corrélation sous les yeux, sans quoi les
   importances des features corrélées seront mal lues